>#                                                             Geospatial Generative AI

![]()

# Create source data

In [0]:
%sql
-- Create the source table in Unity Catalog
CREATE TABLE IF NOT EXISTS workspace.default.airportlocation (
  locationId STRING,
  name       STRING,
  line1      STRING,
  line2      STRING,
  city       STRING,
  state      STRING,
  country    STRING,
  zipCode    STRING
);

-- Clean existing data
TRUNCATE TABLE workspace.default.airportlocation;

-- Insert realistic airport address data
INSERT INTO workspace.default.airportlocation VALUES

-- Pune
('APT001', 'Pune Airport',
 'Lohegaon Airport Terminal',
 'New Airport Road',
 'Pune',
 'Maharashtra',
 'IND',
 '411032'),

-- Mumbai
('APT002', 'Chhatrapati Shivaji Maharaj International Airport',
 'Terminal 2',
 'Sahar Airport Road, Andheri East',
 'Mumbai',
 'Maharashtra',
 'IND',
 '400099'),

-- Delhi
('APT003', 'Indira Gandhi International Airport',
 'Terminal 3',
 'Palam, Airport Road',
 'New Delhi',
 'Delhi',
 'IND',
 '110037'),

-- Bengaluru
('APT004', 'Kempegowda International Airport',
 'KIAL Road',
 'Devanahalli',
 'Bengaluru',
 'Karnataka',
 'IND',
 '560300'),

-- Kolkata
('APT005', 'Netaji Subhas Chandra Bose International Airport',
 'Jessore Road',
 'Dum Dum',
 'Kolkata',
 'West Bengal',
 'IND',
 '700052'),

--Bhubaneswar
 ('APT009', 'Biju Patnaik International Airport',
 'Airport Terminal Building',
 'Aerodrome Area',
 'Bhubaneswar',
 'Odisha',
 'IND',
 '751020'),

-- Hyderabad
('APT006', 'Rajiv Gandhi International Airport',
 'Shamshabad',
 'Airport Main Road',
 'Hyderabad',
 'Telangana',
 'IND',
 '500409'),

-- Chennai
('APT007', 'Chennai International Airport',
 'GST Road',
 'Meenambakkam',
 'Chennai',
 'Tamil Nadu',
 'IND',
 '600027'),

-- Ahmedabad
('APT008', 'Sardar Vallabhbhai Patel International Airport',
 'Hansol',
 'Airport Road',
 'Ahmedabad',
 'Gujarat',
 'IND',
 '380003'),
 
 -- Kochi
 ('APT010', 'Cochin International Airport',
 'Airport Terminal Road',
 'Nedumbassery',
 'Kochi',
 'Kerala',
 'IND',
 '683111');

num_affected_rows,num_inserted_rows
10,10


In [0]:
%sql
select * from workspace.default.airportlocation;

locationId,name,line1,line2,city,state,country,zipCode
APT001,Pune Airport,Lohegaon Airport Terminal,New Airport Road,Pune,Maharashtra,IND,411032
APT002,Chhatrapati Shivaji Maharaj International Airport,Terminal 2,"Sahar Airport Road, Andheri East",Mumbai,Maharashtra,IND,400099
APT003,Indira Gandhi International Airport,Terminal 3,"Palam, Airport Road",New Delhi,Delhi,IND,110037
APT004,Kempegowda International Airport,KIAL Road,Devanahalli,Bengaluru,Karnataka,IND,560300
APT005,Netaji Subhas Chandra Bose International Airport,Jessore Road,Dum Dum,Kolkata,West Bengal,IND,700052
APT009,Biju Patnaik International Airport,Airport Terminal Building,Aerodrome Area,Bhubaneswar,Odisha,IND,751020
APT006,Rajiv Gandhi International Airport,Shamshabad,Airport Main Road,Hyderabad,Telangana,IND,500409
APT007,Chennai International Airport,GST Road,Meenambakkam,Chennai,Tamil Nadu,IND,600027
APT008,Sardar Vallabhbhai Patel International Airport,Hansol,Airport Road,Ahmedabad,Gujarat,IND,380003
APT010,Cochin International Airport,Airport Terminal Road,Nedumbassery,Kochi,Kerala,IND,683111


# Create function for geocoding

In [0]:
%sql
CREATE OR REPLACE FUNCTION workspace.default.geocode_address(
  full_address string
)
RETURNS STRUCT<latitude: DOUBLE, longitude: DOUBLE>
LANGUAGE PYTHON
COMMENT 'Geocode an address using AWS Location Service (geo-places). Returns lat/lng as a struct.'
AS $$
import boto3

try:
    client = boto3.client(
        'geo-places',
        region_name='us-east-1',
        aws_access_key_id='{Put AWS Access Key ID here}',
        aws_secret_access_key='{Put AWS Secret Access Key here}'
    )
    response = client.geocode(QueryText=full_address, MaxResults=1)
    items = response.get('ResultItems', [])
    if items:
        position = items[0]['Position']
        return {"latitude": position[1], "longitude": position[0]}
    return {"latitude": None, "longitude": None}
except Exception:
    return {"latitude": None, "longitude": None}
$$;

In [0]:
%sql
select workspace.default.geocode_address("Purba Bardhaman district, West Bengal, India")

"workspace.default.geocode_address(""Purba Bardhaman district, West Bengal, India"")"
"List(23.24073, 87.86733)"


# Test out the function

In [0]:
%sql
select *, concat(name, ', ',line1, ', ', line2, ', ', city, ', ', state, ', ', country, ', ', zipCode) as full_address,
workspace.default.geocode_address(full_address) as geocoded_data
 from workspace.default.airportlocation;

locationId,name,line1,line2,city,state,country,zipCode,full_address,geocoded_data
APT001,Pune Airport,Lohegaon Airport Terminal,New Airport Road,Pune,Maharashtra,IND,411032,"Pune Airport, Lohegaon Airport Terminal, New Airport Road, Pune, Maharashtra, IND, 411032","List(18.57844, 73.9056)"
APT002,Chhatrapati Shivaji Maharaj International Airport,Terminal 2,"Sahar Airport Road, Andheri East",Mumbai,Maharashtra,IND,400099,"Chhatrapati Shivaji Maharaj International Airport, Terminal 2, Sahar Airport Road, Andheri East, Mumbai, Maharashtra, IND, 400099","List(19.09351, 72.85379)"
APT003,Indira Gandhi International Airport,Terminal 3,"Palam, Airport Road",New Delhi,Delhi,IND,110037,"Indira Gandhi International Airport, Terminal 3, Palam, Airport Road, New Delhi, Delhi, IND, 110037","List(28.55565, 77.08536)"
APT004,Kempegowda International Airport,KIAL Road,Devanahalli,Bengaluru,Karnataka,IND,560300,"Kempegowda International Airport, KIAL Road, Devanahalli, Bengaluru, Karnataka, IND, 560300","List(13.19905, 77.70955)"
APT005,Netaji Subhas Chandra Bose International Airport,Jessore Road,Dum Dum,Kolkata,West Bengal,IND,700052,"Netaji Subhas Chandra Bose International Airport, Jessore Road, Dum Dum, Kolkata, West Bengal, IND, 700052","List(22.64302, 88.43884)"
APT009,Biju Patnaik International Airport,Airport Terminal Building,Aerodrome Area,Bhubaneswar,Odisha,IND,751020,"Biju Patnaik International Airport, Airport Terminal Building, Aerodrome Area, Bhubaneswar, Odisha, IND, 751020","List(20.25511, 85.81585)"
APT006,Rajiv Gandhi International Airport,Shamshabad,Airport Main Road,Hyderabad,Telangana,IND,500409,"Rajiv Gandhi International Airport, Shamshabad, Airport Main Road, Hyderabad, Telangana, IND, 500409","List(17.23652, 78.42938)"
APT007,Chennai International Airport,GST Road,Meenambakkam,Chennai,Tamil Nadu,IND,600027,"Chennai International Airport, GST Road, Meenambakkam, Chennai, Tamil Nadu, IND, 600027","List(12.99317, 80.1838)"
APT008,Sardar Vallabhbhai Patel International Airport,Hansol,Airport Road,Ahmedabad,Gujarat,IND,380003,"Sardar Vallabhbhai Patel International Airport, Hansol, Airport Road, Ahmedabad, Gujarat, IND, 380003","List(23.07796, 72.61844)"
APT010,Cochin International Airport,Airport Terminal Road,Nedumbassery,Kochi,Kerala,IND,683111,"Cochin International Airport, Airport Terminal Road, Nedumbassery, Kochi, Kerala, IND, 683111","List(10.15743, 76.3934)"


# Save the geo-coded data in a table

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.default.airportlocation_geocoded_data AS
select *, concat(name, ', ',line1, ', ', line2, ', ', city, ', ', state, ', ', country, ', ', zipCode) as full_address,
workspace.default.geocode_address(full_address) as geocoded_data 
 from workspace.default.airportlocation;

num_affected_rows,num_inserted_rows


In [0]:
%sql
select * from workspace.default.airportlocation_geocoded_data;

locationId,name,line1,line2,city,state,country,zipCode,full_address,geocoded_data
APT001,Pune Airport,Lohegaon Airport Terminal,New Airport Road,Pune,Maharashtra,IND,411032,"Pune Airport, Lohegaon Airport Terminal, New Airport Road, Pune, Maharashtra, IND, 411032","List(18.57844, 73.9056)"
APT002,Chhatrapati Shivaji Maharaj International Airport,Terminal 2,"Sahar Airport Road, Andheri East",Mumbai,Maharashtra,IND,400099,"Chhatrapati Shivaji Maharaj International Airport, Terminal 2, Sahar Airport Road, Andheri East, Mumbai, Maharashtra, IND, 400099","List(19.09351, 72.85379)"
APT003,Indira Gandhi International Airport,Terminal 3,"Palam, Airport Road",New Delhi,Delhi,IND,110037,"Indira Gandhi International Airport, Terminal 3, Palam, Airport Road, New Delhi, Delhi, IND, 110037","List(28.55565, 77.08536)"
APT004,Kempegowda International Airport,KIAL Road,Devanahalli,Bengaluru,Karnataka,IND,560300,"Kempegowda International Airport, KIAL Road, Devanahalli, Bengaluru, Karnataka, IND, 560300","List(13.19905, 77.70955)"
APT005,Netaji Subhas Chandra Bose International Airport,Jessore Road,Dum Dum,Kolkata,West Bengal,IND,700052,"Netaji Subhas Chandra Bose International Airport, Jessore Road, Dum Dum, Kolkata, West Bengal, IND, 700052","List(22.64302, 88.43884)"
APT009,Biju Patnaik International Airport,Airport Terminal Building,Aerodrome Area,Bhubaneswar,Odisha,IND,751020,"Biju Patnaik International Airport, Airport Terminal Building, Aerodrome Area, Bhubaneswar, Odisha, IND, 751020","List(20.25511, 85.81585)"
APT006,Rajiv Gandhi International Airport,Shamshabad,Airport Main Road,Hyderabad,Telangana,IND,500409,"Rajiv Gandhi International Airport, Shamshabad, Airport Main Road, Hyderabad, Telangana, IND, 500409","List(17.23652, 78.42938)"
APT007,Chennai International Airport,GST Road,Meenambakkam,Chennai,Tamil Nadu,IND,600027,"Chennai International Airport, GST Road, Meenambakkam, Chennai, Tamil Nadu, IND, 600027","List(12.99317, 80.1838)"
APT008,Sardar Vallabhbhai Patel International Airport,Hansol,Airport Road,Ahmedabad,Gujarat,IND,380003,"Sardar Vallabhbhai Patel International Airport, Hansol, Airport Road, Ahmedabad, Gujarat, IND, 380003","List(23.07796, 72.61844)"
APT010,Cochin International Airport,Airport Terminal Road,Nedumbassery,Kochi,Kerala,IND,683111,"Cochin International Airport, Airport Terminal Road, Nedumbassery, Kochi, Kerala, IND, 683111","List(10.15743, 76.3934)"


In [0]:
%sql
CREATE or replace FUNCTION workspace.default.spherical_distance(
 lat1 DOUBLE,
 lng1 DOUBLE,
 lat2 DOUBLE,
 lng2 DOUBLE
)
RETURNS DOUBLE
COMMENT 'Calculates the spherical distance between point 1 (lat1, lng1) and point 2 (lat2, lng2) on earth.'
RETURN (
 6371 -- radius of the earth in km
 * acos(
   sin(radians(lat1)) * sin(radians(lat2))
   + cos(radians(lat1)) * cos(radians(lat2)) * cos(radians(lng1) - radians(lng2))
 )
);

In [0]:
%sql
CREATE OR REPLACE FUNCTION workspace.default.find_nearest_airport(
  input_latitude  DOUBLE COMMENT 'Latitude of the search point',
  input_longitude DOUBLE COMMENT 'Longitude of the search point',
  n INTEGER COMMENT 'No. of rows to return'
)
RETURNS TABLE (
  full_address STRING,
  distance_km DOUBLE
)
COMMENT 'Find nearest airport locations using spherical distance.'
RETURN
SELECT
  full_address,
  distance_km
FROM (
  SELECT
    full_address,
    workspace.default.spherical_distance(
      input_latitude,
      input_longitude,
      geocoded_data.latitude,
      geocoded_data.longitude
    ) AS distance_km,
    ROW_NUMBER() OVER (
      ORDER BY workspace.default.spherical_distance(
        input_latitude,
        input_longitude,
        geocoded_data.latitude,
        geocoded_data.longitude
      ) ASC
    ) AS rn
  FROM workspace.default.airportlocation_geocoded_data
)
WHERE rn <= n;

In [0]:
%sql
select * from workspace.default.find_nearest_airport(19.09351, 72.85379,3) 

full_address,distance_km
"Chhatrapati Shivaji Maharaj International Airport, Terminal 2, Sahar Airport Road, Andheri East, Mumbai, Maharashtra, IND, 400099",0.0
"Pune Airport, Lohegaon Airport Terminal, New Airport Road, Pune, Maharashtra, IND, 411032",124.6311051524223
"Sardar Vallabhbhai Patel International Airport, Hansol, Airport Road, Ahmedabad, Gujarat, IND, 380003",443.72257390879986


In [0]:
%sql
select * from workspace.default.find_nearest_airport(8.0883, 77.5385, 2) 

full_address,distance_km
"Cochin International Airport, Airport Terminal Road, Nedumbassery, Kochi, Kerala, IND, 683111",262.18054010065345
"Kempegowda International Airport, KIAL Road, Devanahalli, Bengaluru, Karnataka, IND, 560300",568.5965900366406
